# Notebook 04e — CMF Stage 3 Static Unlearning (Fixed-Epoch, Unconditional)

**Paper:** *An Illusion of Unlearning? Assessing Machine Unlearning Through Internal Representations*  
(Gao, Unal, Rangamani, Zhu — AISTATS 2026 · arXiv:2604.08271v1)

**Protocol:**

- **Fixed-Epoch Stage 3 Execution:** Runs unconditional fine-tuning using `run_cmf_static` for a fixed epoch budget (`S3_MAX_EPOCHS = 3`) starting from Stage-2 Post-hoc checkpoints.
- **No Gating / No Early Stopping:** Runs all methods across all forget classes unconditionally for the full budget.
- **W stays frozen throughout** via the two-lock mechanism (`requires_grad=False` and monkey-patching `recompute_cmf`).
- Saves checkpoints as `{base_method}_cmf_stage3_static_k{k}_{phase2_data}_{mean_source}_class{forget_class}_seed{seed}.pt`.

In [ ]:
import subprocess, sys
def sh(cmd, verbose=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if verbose and r.stdout: print(r.stdout[-4000:])
    if r.returncode != 0 and r.stderr: print('STDERR:', r.stderr[-2000:])
    return r.returncode
sh('pip install -q timm einops scikit-learn matplotlib seaborn pytorch-lightning torchmetrics')

In [ ]:
import os, sys, json, random, math, time, copy, traceback
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib; import matplotlib.pyplot as plt
matplotlib.rcParams.update({'figure.dpi': 110})
print('PyTorch:', torch.__version__, '  CUDA:', torch.cuda.is_available())

In [ ]:
REPO_DIR = '/kaggle/working/CMF_Unlearning'
if not os.path.isdir(REPO_DIR):
    sh(f'git clone https://github.com/tiensinh2/CMF_Unlearning.git {REPO_DIR}')
else:
    sh(f'git -C {REPO_DIR} remote set-url origin https://github.com/tiensinh2/CMF_Unlearning.git')
    sh(f'git -C {REPO_DIR} pull origin main')
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
result = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'],
                        capture_output=True, text=True)
REPO_COMMIT = result.stdout.strip() or 'main'
print('Repo commit:', REPO_COMMIT)

In [ ]:
CKPT_DATASET_DIR = '/kaggle/input/datasets/kiethe/cmf-nb1-cifar100'
_CONFIG_CANDIDATES = [
    f'{CKPT_DATASET_DIR}/cmf_benchmark_config.json',
    f'{CKPT_DATASET_DIR}/checkpoints/cmf_benchmark/cmf_benchmark_config.json',
    f'{CKPT_DATASET_DIR}/cmf_benchmark/cmf_benchmark_config.json',
    './notebooks/Result_nb1/checkpoints/cmf_benchmark/cmf_benchmark_config.json',
]
config_path = CKPT_ROOT_NB1 = None
for _p in _CONFIG_CANDIDATES:
    if os.path.exists(_p):
        config_path = _p; CKPT_ROOT_NB1 = os.path.dirname(_p); break

if config_path and os.path.exists(config_path):
    with open(config_path) as f: NB1_CFG = json.load(f)
    DATASET     = NB1_CFG.get('dataset', 'cifar100')
    ARCH        = NB1_CFG.get('arch', 'resnet18')
    NUM_CLASSES = NB1_CFG.get('num_classes', 100)
    TEST_MODE   = NB1_CFG.get('test_mode', False)
else:
    DATASET     = 'cifar100'
    ARCH        = 'resnet18'
    NUM_CLASSES = 100
    TEST_MODE   = False
    CKPT_ROOT_NB1 = '/kaggle/working/checkpoints/cmf_benchmark'

STAGE = '4e'
# Paper §A.3 / Table 1&3: CIFAR-100 uses 5 single-class forget sets
# config.py UNLEARN_SCENARIOS['cifar100']['single'] = [[0],[1],[2],[3],[5]]
FORGET_CLASSES = [0, 1, 2, 3, 5]   # paper CIFAR-100 single-class scenario
SEEDS          = [0]
BASE_METHODS   = ['grad_ascent_descent', 'random_label', 'salun', 'scrub', 'tarun']
MEAN_SOURCES   = ['train']

# Stage-2 (post-hoc) inputs to load from NB4b.
K_POSTHOC   = 2
PHASE2_DATA = 'retain_only'

# ── Stage-3 static unlearning hyperparameters ────────────────────────────────
# LR = Static LR / 5
S3_LR = {
    'random_label':        4e-4,   # 2e-3 / 5
    'salun':               4e-4,   # 2e-3 / 5
    'grad_ascent_descent': 2e-5,   # 1e-4 / 5
    'scrub':               1e-3,   # 5e-3 / 5
    'tarun':               1e-5,   # 5e-5 / 5
}
S3_BATCH        = {'scrub': 64}
S3_BATCH_DEFAULT = 128
S3_MAX_EPOCHS   = 1 if TEST_MODE else 3   # fixed epoch budget

# Paths — NB4b checkpoint root (attach as Kaggle input dataset).
CKPT_ROOT_NB4B = '/kaggle/input/datasets/nguyenhunguet/4b-cmf/checkpoints/cmf_posthoc_paper'
CKPT_ROOT      = '/kaggle/working/checkpoints/cmf_stage3_static'
os.makedirs(f'{CKPT_ROOT}/cmf_stage3_static', exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'STAGE={STAGE}  DATASET={DATASET}  ARCH={ARCH}  device={device}')
print(f'S3_MAX_EPOCHS={S3_MAX_EPOCHS}')
print(f'CKPT_ROOT_NB4B exists: {os.path.isdir(CKPT_ROOT_NB4B)}')

In [ ]:
import torchvision, torchvision.transforms as transforms
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

full_train      = torchvision.datasets.CIFAR100('/kaggle/working/data', train=True,
                                                download=True,  transform=transform_train)
full_train_eval = torchvision.datasets.CIFAR100('/kaggle/working/data', train=True,
                                                download=False, transform=transform_test)
test_set        = torchvision.datasets.CIFAR100('/kaggle/working/data', train=False,
                                                download=True, transform=transform_test)

test_targets  = torch.tensor(test_set.targets)
TEST_CLASS_IDX  = {
    c: (test_targets == c).nonzero(as_tuple=True)[0].tolist()
    for c in range(NUM_CLASSES)
}
train_targets = torch.tensor(full_train.targets)
TRAIN_CLASS_IDX = {
    c: (train_targets == c).nonzero(as_tuple=True)[0].tolist()
    for c in range(NUM_CLASSES)
}
print(f'Train size: {len(full_train)}  Test size: {len(test_set)}')

In [ ]:
import argparse
from unlearn.cmf_weights import ModelModule
from unlearn.cmf_two_stage import run_cmf_static

def build_cmf_model(args):
    model = ModelModule(args).to(device)
    assert hasattr(model, 'CMFweights'), 'Model must have CMFweights attribute!'
    return model

def make_cmf_args(base_method, lr, epochs, mean_source, forget_class,
                  forget_train_idx, retain_train_idx, seed=0):
    return argparse.Namespace(
        dataset=DATASET, arch=ARCH, num_classes=NUM_CLASSES,
        class_label_names=list(range(NUM_CLASSES)),
        unlearn_method=f'{base_method}_CMF_RemoveFC',
        unlearn_class=[forget_class],
        batch_size=128, test_batch_size=256, lr=lr,
        momentum=0.9, weight_decay=5e-4, epochs_or_steps=epochs,
        seed=seed,
        num_retain_samples=len(retain_train_idx),
        num_forget_samples=len(forget_train_idx),
        grad_norm_clip=1.0,
        SVD_alpha_r=1000, SVD_alpha_f=30,
        SVD_samples=900, SVD_max_patches=10000,
        freeze_except_last=False,
        scrub_del_bsz=64, scrub_sgda_bsz=64, scrub_msteps=2, scrub_epochs=epochs,
        salun_threshold=0.5,
        tarun_impair_lr=lr, tarun_samples_per_class=1000,
        dry_run=TEST_MODE, no_cuda=False, no_mps=True, gamma=0.5,
        data_path='/kaggle/working/data', remove_FC=True,
        CMFClassifier=True, CMF_momentum=0.9, pretrained=False, temperature=1.0,
        prob_batch_size=256, lp_every=0, mean_source=mean_source,
        repo_commit=REPO_COMMIT, test_mode=TEST_MODE,
    )

@torch.no_grad()
def eval_acc(model, loader):
    model.eval()
    correct = total = 0
    for x, y in loader:
        correct += (model(x.to(device)).argmax(1).cpu() == y).sum().item()
        total   += y.size(0)
    return 100.0 * correct / max(total, 1)

@torch.no_grad()
def cmf_extract_features(model, loader):
    model.eval()
    feats, labs = [], []
    for x, y in loader:
        x = x.to(device)
        f = model.extract_features(x)
        z = model.extract_features(x)
        feats.append(z.cpu()); labs.append(y)
    return torch.cat(feats), torch.cat(labs)

def run_probe_cmf(model, train_retain_ldr, train_forget_ldr,
                  test_retain_ldr, test_forget_ldr, n_epochs=None, seed=42):
    if n_epochs is None:
        n_epochs = 200 if DATASET.lower() == 'cifar100' else 50
    torch.manual_seed(seed); np.random.seed(seed)
    Xtr, ytr = cmf_extract_features(model, train_retain_ldr)
    Xfg, yfg = cmf_extract_features(model, train_forget_ldr)
    Xall = torch.cat([Xtr, Xfg]); yall = torch.cat([ytr, yfg])
    head = nn.Linear(Xall.size(1), NUM_CLASSES).to(device)
    opt  = optim.SGD(head.parameters(), lr=1e-2, momentum=0.9)
    ldr  = torch.utils.data.DataLoader(
               torch.utils.data.TensorDataset(Xall, yall), batch_size=256, shuffle=True)
    for _ in range(n_epochs):
        head.train()
        for bx, by in ldr:
            opt.zero_grad()
            F.cross_entropy(head(bx.to(device)), by.to(device)).backward()
            opt.step()
    head.eval()
    with torch.no_grad():
        Xte_r, yte_r = cmf_extract_features(model, test_retain_ldr)
        Xte_f, yte_f = cmf_extract_features(model, test_forget_ldr)
        ret_acc = (head(Xte_r.to(device)).argmax(1).cpu() == yte_r).float().mean().item() * 100
        fgt_acc = (head(Xte_f.to(device)).argmax(1).cpu() == yte_f).float().mean().item() * 100
    return ret_acc, fgt_acc

def run_ncc_cmf(model, train_retain_ldr, train_forget_ldr,
                test_retain_ldr, test_forget_ldr):
    Xtr, ytr = cmf_extract_features(model, train_retain_ldr)
    Xfg, yfg = cmf_extract_features(model, train_forget_ldr)
    Xall = torch.cat([Xtr, Xfg]); yall = torch.cat([ytr, yfg])
    means = []
    for c in range(NUM_CLASSES):
        mask = (yall == c)
        mu = Xall[mask].mean(0) if mask.any() else torch.zeros(Xall.size(1))
        means.append(mu)
    M = torch.stack(means)
    Xte_r, yte_r = cmf_extract_features(model, test_retain_ldr)
    Xte_f, yte_f = cmf_extract_features(model, test_forget_ldr)
    ret_pred = torch.cdist(Xte_r.unsqueeze(0), M.unsqueeze(0)).squeeze(0).argmin(1)
    fgt_pred = torch.cdist(Xte_f.unsqueeze(0), M.unsqueeze(0)).squeeze(0).argmin(1)
    return ((ret_pred == yte_r).float().mean().item() * 100,
            (fgt_pred == yte_f).float().mean().item() * 100)

def eval_cmf_three_metrics(model, test_retain_ldr, test_forget_ldr,
                           train_retain_eval_ldr, train_forget_eval_ldr):
    out_ret = eval_acc(model, test_retain_ldr)
    out_fgt = eval_acc(model, test_forget_ldr)
    lp_ret, lp_fgt   = run_probe_cmf(model, train_retain_eval_ldr, train_forget_eval_ldr,
                                      test_retain_ldr, test_forget_ldr)
    ncc_ret, ncc_fgt = run_ncc_cmf(model, train_retain_eval_ldr, train_forget_eval_ldr,
                                    test_retain_ldr, test_forget_ldr)
    return {'output_retain_acc': out_ret, 'output_forget_acc': out_fgt,
            'probe_retain_acc':  lp_ret,  'probe_forget_acc':  lp_fgt,
            'ncc_retain_acc':    ncc_ret, 'ncc_forget_acc':    ncc_fgt}

print('Helpers ready.')

In [ ]:
def freeze_cmf_weights(model):
    """Hard-freeze CMFweights.weight so run_cmf_static cannot overwrite it."""
    model.CMFweights.weight.requires_grad_(False)
    if not hasattr(model, '_recompute_cmf_orig'):
        model._recompute_cmf_orig = model.recompute_cmf
    model.recompute_cmf = lambda *a, **kw: None   # no-op
    print('[freeze_cmf_weights] CMFweights.weight frozen; recompute_cmf no-op.')

def unfreeze_cmf_weights(model):
    """Restore recompute_cmf to its original implementation (for final eval)."""
    if hasattr(model, '_recompute_cmf_orig'):
        model.recompute_cmf = model._recompute_cmf_orig
        del model._recompute_cmf_orig
    print('[unfreeze_cmf_weights] recompute_cmf restored.')

print('freeze_cmf_weights helper ready.')

In [ ]:
results_4e = []

for forget_class in FORGET_CLASSES:
    for seed in SEEDS:
        forget_train_idx = TRAIN_CLASS_IDX[forget_class]
        retain_train_idx = [
            i for c in range(NUM_CLASSES)
            if c != forget_class
            for i in TRAIN_CLASS_IDX[c]
        ]
        retain_loader = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train, retain_train_idx),
            batch_size=128, shuffle=True, num_workers=2)
        forget_loader = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train, forget_train_idx),
            batch_size=128, shuffle=True, num_workers=2)
        full_train_loader = torch.utils.data.DataLoader(
            full_train, batch_size=256, shuffle=False, num_workers=2)
        train_retain_eval_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train_eval, retain_train_idx),
            batch_size=256, shuffle=False, num_workers=2)
        train_forget_eval_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train_eval, forget_train_idx),
            batch_size=256, shuffle=False, num_workers=2)
        test_forget_idx = TEST_CLASS_IDX[forget_class]
        test_retain_idx = [
            i for c in range(NUM_CLASSES)
            if c != forget_class
            for i in TEST_CLASS_IDX[c]
        ]
        test_forget_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(test_set, test_forget_idx),
            batch_size=256, shuffle=False, num_workers=2)
        test_retain_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(test_set, test_retain_idx),
            batch_size=256, shuffle=False, num_workers=2)
        test_loader_full = torch.utils.data.DataLoader(
            test_set, batch_size=256, shuffle=False, num_workers=2)

        for base_method in BASE_METHODS:
            for mean_source in MEAN_SOURCES:

                # ── 1. Locate and load the Stage-2 (post-hoc) checkpoint ────────────
                s2_tag = (f'{base_method}_cmf_posthoc_k{K_POSTHOC}_{PHASE2_DATA}'
                          f'_{mean_source}_class{forget_class}_seed{seed}')
                if TEST_MODE: s2_tag += '_testmode'
                s2_candidates = [
                    f'{CKPT_ROOT_NB4B}/cmf_posthoc/{s2_tag}.pt',
                    f'{CKPT_ROOT_NB4B}/{s2_tag}.pt',
                    f'./notebooks/Result_nb4_postdoc/checkpoints/{s2_tag}.pt',
                    f'./checkpoints/cmf_posthoc_paper/cmf_posthoc/{s2_tag}.pt',
                ]
                s2_path = next((p for p in s2_candidates if os.path.exists(p)), None)
                if s2_path is None:
                    print(f'Stage-2 checkpoint not found for {s2_tag}. Skipping.')
                    continue

                print(f'\n[Loaded S2] {s2_path}')
                ck_s2    = torch.load(s2_path, map_location=device)
                s2_state = ck_s2.get('model_state_dict', ck_s2)

                s3_epochs = S3_MAX_EPOCHS
                lr        = S3_LR.get(base_method, 1e-3)
                batch     = S3_BATCH.get(base_method, S3_BATCH_DEFAULT)

                tag = (f'{base_method}_cmf_stage3_static_k{K_POSTHOC}_{PHASE2_DATA}'
                       f'_{mean_source}_class{forget_class}_seed{seed}')
                if TEST_MODE: tag += '_testmode'
                ckpt_path = f'{CKPT_ROOT}/cmf_stage3_static/{tag}.pt'

                if os.path.exists(ckpt_path):
                    print(f'[{tag}] exists — loading cached result.')
                    ck = torch.load(ckpt_path, map_location=device)
                    results_4e.append(ck['metrics'])
                    continue

                print(f'\n[{tag}] Running Stage 3 Static ({base_method}, {s3_epochs} epochs, W frozen)...')
                torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)

                args = make_cmf_args(base_method, lr, s3_epochs, mean_source,
                                     forget_class, forget_train_idx, retain_train_idx,
                                     seed=seed)
                args.batch_size = batch
                if base_method == 'scrub':
                    args.scrub_del_bsz  = 64
                    args.scrub_sgda_bsz = 64

                # Build model and load Stage-2 weights (encoder + frozen W).
                model = build_cmf_model(args)
                model.load_state_dict(s2_state, strict=False)

                # ── Baseline metrics before Stage 3 ──────────────────────────────
                metrics_before = eval_cmf_three_metrics(
                    model, test_retain_ldr, test_forget_ldr,
                    train_retain_eval_ldr, train_forget_eval_ldr
                )
                print(f'  [Before S3] out R={metrics_before["output_retain_acc"]:.2f}% '
                      f'F={metrics_before["output_forget_acc"]:.2f}%  '
                      f'ncc F={metrics_before["ncc_forget_acc"]:.2f}%')

                # ── FREEZE W ─────────────────────────────────────────────────────
                freeze_cmf_weights(model)
                for name, p in model.named_parameters():
                    if 'CMFweights' not in name:
                        p.requires_grad_(True)

                mean_loader = full_train_loader if mean_source == 'train' else retain_loader
                t0 = time.time()
                try:
                    # Run standard fixed-epoch CMF unlearning with W frozen
                    model = run_cmf_static(
                        base_method=base_method,
                        args=args,
                        model=model,
                        device=device,
                        retain_loader=retain_loader,
                        forget_loader=forget_loader,
                        train_loader=mean_loader,
                        test_loader=test_loader_full,
                        epochs=s3_epochs,
                        test_forget_loader=test_forget_ldr,
                        train_dataset=full_train,
                        val_index=retain_train_idx,
                    )
                except Exception as e:
                    print(f'ERROR during Stage 3 ({base_method}): {e}')
                    traceback.print_exc()
                    unfreeze_cmf_weights(model)
                    continue

                wall_min = (time.time() - t0) / 60
                unfreeze_cmf_weights(model)
                model.eval()

                # ── Metrics after Stage 3 ─────────────────────────────────────────
                metrics = eval_cmf_three_metrics(
                    model, test_retain_ldr, test_forget_ldr,
                    train_retain_eval_ldr, train_forget_eval_ldr
                )

                # ── Sanity check: W must be identical before and after Stage 3 ────
                w_before = s2_state.get('CMFweights.weight',
                           s2_state.get('encoder.CMFweights.weight', None))
                w_after  = model.CMFweights.weight.detach().cpu()
                if w_before is not None:
                    w_max_diff = (w_after - w_before.cpu()).abs().max().item()
                    print(f'  [W sanity] max |W_after - W_before| = {w_max_diff:.6f}'
                          f'  (expected ~0.0)')
                    if w_max_diff > 1e-5:
                        print('  WARNING: W changed during Stage 3! Check freeze logic.')
                        metrics['w_invariance_violated'] = True
                    else:
                        metrics['w_invariance_violated'] = False
                else:
                    metrics['w_invariance_violated'] = None

                metrics.update({
                    'method': base_method, 'mean_source': mean_source,
                    'forget_class': forget_class, 'seed': seed, 'stage': STAGE,
                    's3_epochs': s3_epochs, 'lr': lr, 'batch': batch,
                    'k_posthoc': K_POSTHOC, 'phase2_data': PHASE2_DATA,
                    'wall_clock_minutes': wall_min,
                    'n_forget_train': len(forget_train_idx),
                    'n_retain_train': len(retain_train_idx),
                    'protocol': 'whole_class_single',
                    'before_output_retain_acc': metrics_before['output_retain_acc'],
                    'before_output_forget_acc': metrics_before['output_forget_acc'],
                    'before_ncc_forget_acc':    metrics_before['ncc_forget_acc'],
                })

                torch.save({
                    'model_state_dict': model.state_dict(),
                    'config': {
                        'base_method': base_method, 'mean_source': mean_source,
                        'dataset': DATASET, 'arch': ARCH, 'num_classes': NUM_CLASSES,
                        'forget_class': forget_class, 'seed': seed,
                        's3_epochs': s3_epochs, 'lr': lr, 'batch': batch,
                        'k_posthoc': K_POSTHOC, 'phase2_data': PHASE2_DATA,
                        'protocol': 'whole_class_single',
                        'repo_commit': REPO_COMMIT, 'test_mode': TEST_MODE,
                    },
                    'seed': seed, 'metrics': metrics,
                }, ckpt_path)
                print(f'  Saved {ckpt_path}')
                print(f'  out   R={metrics["output_retain_acc"]:.2f}%  '
                      f'F={metrics["output_forget_acc"]:.2f}%')
                print(f'  probe R={metrics["probe_retain_acc"]:.2f}%  '
                      f'F={metrics["probe_forget_acc"]:.2f}%')
                print(f'  ncc   R={metrics["ncc_retain_acc"]:.2f}%  '
                      f'F={metrics["ncc_forget_acc"]:.2f}%')
                delta_out = metrics['output_forget_acc'] - metrics_before['output_forget_acc']
                delta_ncc = metrics['ncc_forget_acc']    - metrics_before['ncc_forget_acc']
                print(f'  delta_output_forget={delta_out:+.2f}%  '
                      f'delta_ncc_forget={delta_ncc:+.2f}%')
                results_4e.append(metrics)

df_4e = pd.DataFrame(results_4e)
csv_path = f'{CKPT_ROOT}/results_4e_cmf_stage3_static.csv'
df_4e.to_csv(csv_path, index=False)
print(f'\nResults saved to {csv_path}')
if not df_4e.empty:
    metric_cols = ['output_retain_acc', 'output_forget_acc',
                   'probe_retain_acc',  'probe_forget_acc',
                   'ncc_retain_acc',    'ncc_forget_acc']
    print('\n=== Summary (mean over forget classes) ===')
    print(df_4e.groupby('method')[metric_cols].mean().round(2).to_string())
    print('\n=== Delta vs Stage-2 baseline (output forget acc) ===')
    df_4e['delta_output_forget'] = df_4e['output_forget_acc'] - df_4e['before_output_forget_acc']
    df_4e['delta_ncc_forget']    = df_4e['ncc_forget_acc']    - df_4e['before_ncc_forget_acc']
    print(df_4e.groupby('method')[['delta_output_forget','delta_ncc_forget']].mean().round(2).to_string())